# Ćwiczenia 6 — Pierwsze klasyfikatory: regresja logistyczna i naiwny Bayes (wersja Python, z rozwiązaniami)
**Dane przykładowe: kohorta T2D (QinJ_2012)**

Na Ćwiczeniach 4 powstał pierwszy model, k najbliższych sąsiadów (k-NN), porównany z klasyfikatorem bazowym, a na Ćwiczeniach 5 nauczyliśmy się oceniać modele w walidacji krzyżowej. Na tym spotkaniu dochodzą dwa modele, które klasyfikują próbki inaczej niż k-NN: regresja logistyczna i naiwny klasyfikator Bayesa. Wszystkie modele oceniamy w tych samych częściach walidacji krzyżowej i porównujemy. Każdy nowy model to w scikit-learn ten sam potok, w którym zmienia się tylko ostatni krok.

Zbiór testowy odkładamy i nie używamy go: zostaje do oceny końcowej na Ćwiczeniach 9. Notatnik najlepiej uruchamiać od początku po ponownym uruchomieniu jądra (*Kernel → Restart*), komórka po komórce. Liczby różnią się nieco od wersji R, bo oba języki inaczej losują podział; wnioski są takie same.

## Punkt wyjścia

Przygotowanie danych jak na Ćwiczeniach 4 i 5: podział warstwowy z ziarnem 42, filtr częstości występowania i CLR. Potem raz ustalamy pięć części walidacji krzyżowej; wszystkie modele korzystają z tych samych części, więc różnice wyników wynikają z modeli, a nie z losowania części.

Cech jest prawie tyle, ile próbek treningowych, a przy takim kształcie danych regresja logistyczna nie daje stabilnych wyników. Dlatego modele otrzymują jako cechy 10 pierwszych składowych PCA (Ćwiczenia 3). Standaryzacja i PCA są krokami potoku, więc `cross_val_score()` wyznacza je w każdej części uczącej osobno i nie ma przecieku (Ćwiczenia 5).

In [ ]:
import numpy as np                       # obliczenia na tablicach
import pandas as pd                      # tabele danych
import matplotlib.pyplot as plt          # wykresy
from sklearn.model_selection import train_test_split, StratifiedKFold   # podział i części walidacji
from sklearn.model_selection import cross_val_score, GridSearchCV       # walidacja krzyżowa i strojenie
from sklearn.pipeline import make_pipeline             # potok: kroki i model w jednym obiekcie
from sklearn.preprocessing import StandardScaler       # standaryzacja
from sklearn.decomposition import PCA                  # analiza głównych składowych
from sklearn.dummy import DummyClassifier              # klasyfikator bazowy
from sklearn.neighbors import KNeighborsClassifier     # k najbliższych sąsiadów
from sklearn.linear_model import LogisticRegression    # regresja logistyczna
from sklearn.naive_bayes import GaussianNB             # naiwny Bayes z rozkładami normalnymi
pd.set_option("display.width", 120)      # szersze wydruki tabel

X = pd.read_csv("dane/t2d_features.csv", index_col=0)   # index_col=0: pierwsza kolumna to nazwy próbek
# ["grupa"]: kolumna z etykietami; index=X.index: te same nazwy wierszy co w X
y = pd.Series(pd.read_csv("dane/t2d_labels.csv")["grupa"].to_numpy(), index=X.index)
# test_size=0.25: 25% próbek do testu; stratify=y: osobno w każdej klasie; random_state=42: ziarno kursu
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

prevalence_tr = (X_tr > 0).mean()        # częstość występowania każdej cechy w treningu
keep = prevalence_tr.index[prevalence_tr >= 0.10]   # nazwy cech obecnych w co najmniej 10% próbek

# CLR: logarytm (z dodaną stałą 1e-6, bo log(0) nie istnieje) minus średnia logarytmów w próbce
def clr(M):
    L = np.log(M + 1e-6)                 # logarytm każdej wartości
    return L.sub(L.mean(axis=1), axis=0) # od każdego wiersza (próbki) odejmujemy jego średnią

Z_tr = clr(X_tr[keep])                   # cechy treningowe po filtrze i CLR
print(Z_tr.shape)                        # liczba próbek treningowych i liczba cech

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)   # te same 5 części dla wszystkich modeli

## 1. Punkty odniesienia: klasyfikator bazowy i k-NN

Nowe modele trzeba z czymś porównać. Klasyfikator bazowy zawsze wskazuje klasę liczniejszą i wyznacza poziom, który każdy model musi przekroczyć. k-NN z Ćwiczeń 4 i 5 jest modelem, który już znamy; tu dobieramy liczbę sąsiadów tak jak na Ćwiczeniach 5.

In [ ]:
acc_base = cross_val_score(DummyClassifier(strategy="most_frequent"), Z_tr, y_tr, cv=cv)   # klasyfikator bazowy
print(acc_base.mean(), acc_base.std(ddof=1))   # średnia dokładność z 5 części i odchylenie standardowe

pipe_knn = make_pipeline(StandardScaler(), PCA(n_components=10), KNeighborsClassifier())   # standaryzacja, 10 składowych, k-NN
grid = {"kneighborsclassifier__n_neighbors": list(range(1, 26, 2))}   # sprawdzane wartości k
fit_knn = GridSearchCV(pipe_knn, grid, cv=cv).fit(Z_tr, y_tr)         # walidacja dla każdego k
print(fit_knn.best_params_)              # k o najwyższej średniej dokładności
print(fit_knn.best_score_)               # średnia dokładność przy tym k

**Na co patrzeć.** Klasyfikator bazowy osiąga około 0,5. Każdy kolejny wynik porównuje się z tą wartością i z wynikiem k-NN.

## 2. Regresja logistyczna

Regresja logistyczna (ang. *logistic regression*) liczy dla każdej próbki prawdopodobieństwo klasy T2D: sumuje cechy pomnożone przez współczynniki (wagi) i zamienia tę sumę na liczbę od 0 do 1. Próbkę przypisuje się do T2D, gdy prawdopodobieństwo wynosi co najmniej 0,5. Względem k-NN w potoku zmienia się tylko ostatni krok; `C=np.inf` wyłącza karę, którą scikit-learn dodaje domyślnie (karę poznamy na Ćwiczeniach 9).

In [ ]:
n_comp = 10                              # liczba składowych PCA (do zmiany w 🔧)
pipe_glm = make_pipeline(StandardScaler(), PCA(n_components=n_comp), LogisticRegression(C=np.inf))   # regresja logistyczna
acc_glm = cross_val_score(pipe_glm, Z_tr, y_tr, cv=cv)   # te same części
print(acc_glm.mean(), acc_glm.std(ddof=1))   # średnia dokładność z 5 części i odchylenie standardowe

pipe_glm.fit(Z_tr, y_tr)                 # model końcowy: dopasowanie na całym treningu
# prawdopodobieństwa klas dla pierwszych 6 próbek; kolumny w kolejności pipe_glm.classes_
print(pd.DataFrame(pipe_glm.predict_proba(Z_tr.iloc[:6]), columns=pipe_glm.classes_, index=Z_tr.index[:6]).round(3))
print(pipe_glm.predict(Z_tr.iloc[:6]))   # przewidziana klasa: ta o większym prawdopodobieństwie

**Na co patrzeć.** Model zwraca dla każdej próbki dwa prawdopodobieństwa, które sumują się do 1, i klasę o większym z nich. Prawdopodobieństwa przydadzą się na Ćwiczeniach 7, gdzie próg 0,5 będzie można zmieniać.

🔧 **Do sprawdzenia.** Zmienić `n_comp = 10` na `5`, a potem na `20`, i ponownie uruchomić komórkę. Jak zmienia się średnia dokładność?

> **✏️ Zadanie 1.** Policzyć dokładność regresji logistycznej na zbiorze treningowym (metoda `score()` potoku `pipe_glm` na `Z_tr` i `y_tr`) i porównać ją z wynikiem walidacji krzyżowej. Który wynik jest wyższy i dlaczego do oceny modelu nadaje się tylko drugi?

In [ ]:
print(pipe_glm.score(Z_tr, y_tr))       # dokładność na treningu

**Rozwiązanie.** Dokładność na treningu wynosi 0,628, a w walidacji krzyżowej 0,605: wynik na treningu
jest wyższy, ale tylko o około 2 punkty procentowe. Na treningu model ocenia się na tych samych próbkach, na których go
uczono, więc wynik jest zawyżony. Walidacja krzyżowa sprawdza model na próbkach, których w danym
powtórzeniu nie użyto do uczenia, i tylko ona mówi, jak model poradzi sobie z nowymi danymi. Tu różnica
jest mała, bo regresja logistyczna na 10 składowych jest prostym modelem i w niewielkim stopniu ulega
przeuczeniu (inaczej niż k-NN z jednym sąsiadem na Ćwiczeniach 4, który na treningu miał dokładność 1).

## 3. Naiwny klasyfikator Bayesa

Naiwny klasyfikator Bayesa (ang. *naive Bayes*) opisuje każdą cechę w każdej klasie osobno (tu rozkładem normalnym: średnia i odchylenie standardowe w klasie) i przypisuje nową próbkę do klasy, do której jej cechy pasują lepiej. „Naiwność” polega na założeniu, że cechy w obrębie klasy są od siebie niezależne. Składowe PCA są ze sobą nieskorelowane, więc dla nich założenie jest mniej naruszone niż dla surowych cech.

In [ ]:
pipe_nb = make_pipeline(StandardScaler(), PCA(n_components=10), GaussianNB())   # naiwny Bayes na 10 składowych
acc_nb = cross_val_score(pipe_nb, Z_tr, y_tr, cv=cv)   # te same części
print(acc_nb.mean(), acc_nb.std(ddof=1))   # średnia dokładność z 5 części i odchylenie standardowe

> **✏️ Zadanie 2.** Ocenić naiwny Bayes na wszystkich cechach CLR, bez PCA (potok `make_pipeline(StandardScaler(), GaussianNB())`). Jak zmienia się średnia dokładność? Założenie o niezależności cech jest wtedy mocniej naruszone; czy to musi pogarszać wynik?

In [ ]:
a = cross_val_score(make_pipeline(StandardScaler(), GaussianNB()), Z_tr, y_tr, cv=cv)   # bez PCA, te same części
print(a.mean(), a.std(ddof=1))          # średnia dokładność i odchylenie standardowe

**Rozwiązanie.** Na wszystkich cechach CLR średnia dokładność wynosi 0,624 (SD 0,025), a na 10 składowych PCA
0,608 (SD 0,086): bez PCA wynik jest wyższy o około 2 punkty procentowe, choć cechy CLR są ze sobą skorelowane i założenie
o niezależności jest mocniej naruszone. Naruszenie założenia nie musi pogarszać klasyfikacji: model źle
szacuje prawdopodobieństwa, ale klasa o większym prawdopodobieństwie bywa nadal właściwa. Poza tym
10 składowych PCA pomija część informacji, którą naiwny Bayes może wykorzystać na pełnych cechach.
O wyborze modelu decyduje wynik walidacji, a nie samo spełnienie założeń.

## 4. Porównanie modeli

Cztery modele oceniano na tych samych pięciu częściach, więc można porównać ich wyniki część po części. Jeśli jeden model jest lepszy w każdej części, przewaga jest wiarygodna; jeśli jest lepszy tylko średnio, różnica może wynikać z przypadku. Wynik k-NN w każdej części liczymy jeszcze raz, z wybraną liczbą sąsiadów.

In [ ]:
acc_knn = cross_val_score(fit_knn.best_estimator_, Z_tr, y_tr, cv=cv)   # k-NN z najlepszym k w tych samych częściach
acc = pd.DataFrame({"bazowy": acc_base, "kNN": acc_knn, "logistyczna": acc_glm, "Bayes": acc_nb},   # kolumny: modele
                   index=[f"Fold{i}" for i in range(1, 6)])   # wiersze: części
print(acc.round(3))                      # dokładność każdego modelu w każdej części
print(acc.describe().loc[["min", "mean", "max"]].round(3))   # najmniejsza, średnia i największa dokładność

acc.plot.box()                           # wykres pudełkowy: pięć wyników każdego modelu
plt.ylabel("dokładność w części")        # podpis osi Y
plt.show()                               # wyświetlenie wykresu

**Na co patrzeć.** Każde pudełko to pięć wyników jednego modelu. Różnica średnich mniejsza od rozrzutu wyników między częściami nie wskazuje jednoznacznie lepszego modelu.

> **✏️ Zadanie 3.** Policzyć, w ilu z pięciu części regresja logistyczna ma wyższą dokładność niż k-NN (`(acc["logistyczna"] > acc["kNN"]).sum()`). Czy na tej podstawie można wskazać lepszy model?

In [ ]:
print((acc["logistyczna"] > acc["kNN"]).sum())   # części, w których regresja jest lepsza

**Rozwiązanie.** Regresja logistyczna ma wyższą dokładność tylko w jednej części z pięciu, a k-NN ma
średnią wyższą o 1,5 punktu procentowego (0,620 wobec 0,605). Ta różnica jest porównywalna
z rozrzutem wyników między częściami (odchylenie standardowe około 0,02–0,03), a w wersji R wynik
wypada odwrotnie. Nie można więc wskazać wyraźnie lepszego modelu. Przy podobnym wyniku wybiera się
model prostszy albo łatwiejszy do interpretacji.